In [1]:
from array import array
from collections import deque, defaultdict, Counter, namedtuple

# Student Identity
LAST_NAME = "PALAOAG"
STUDENT_ID = "TUPM-26-1949"
FAVORITE_ARTIST = "RICO BLANCO"
SEED_NUM = int(STUDENT_ID[-1])

# Identity-Based Computational Seed (immutable tuples)
LAST_NAME    = LAST_NAME.upper().replace(" ", "")
SEED_DIGITS = (SEED_NUM,)
ARTIST_CODES = tuple(ord(ch) for ch in FAVORITE_ARTIST.upper() if ch.isalpha())
IDENTITY_SEED = sum(SEED_DIGITS) * len(ARTIST_CODES) + sum(c * (i + 1) for i, c in enumerate(ARTIST_CODES))

def make_stream():
    """Fresh number generator. Two deques are rotated on every call, so the
    SEED_NUM digits and FAVORITE_ARTIST character positions keep cycling."""
    digit_ring = deque(SEED_DIGITS)
    code_ring = deque(ARTIST_CODES)
    state = {"step": 0}

    def next_value(lo, hi):
        step = state["step"]
        pos = step % len(code_ring) + 1        
        n = (digit_ring[0] + 1) * code_ring[0] * pos + code_ring[-1] + step * 7 + IDENTITY_SEED % 13
        digit_ring.rotate(-1)
        code_ring.rotate(-1)
        state["step"] += 1
        return lo + n % (hi - lo + 1)

    return next_value

def table(columns, rows):
    line = " | ".join(f"{title:<{w}}" for title, w in columns)
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(f"{str(v):<{w}}" for v, (_, w) in zip(r, columns)))

def mean(values):
    return sum(values) / len(values) if len(values) else 0.0

print("IDENTITY_SEED =", IDENTITY_SEED)
print("SEED_DIGITS   =", SEED_DIGITS)

IDENTITY_SEED = 4093
SEED_DIGITS   = (9,)


In [ ]:
Score = namedtuple("Score", "student subject score")           

students = tuple(f"{LAST_NAME}_S{i}" for i in range(1, 6))
subjects = ("PROGRAMMING", "MATHEMATICS", "ELECTRONICS", "PHYSICS", "ENGLISH")

GRADE_BANDS = ((90, "EXCELLENT"), (80, "GOOD"), (75, "SATISFACTORY"), (0, "NEEDS IMPROVEMENT"))

def categorize(score):
    if score is None:
        return "INCOMPLETE"
    return next(label for cutoff, label in GRADE_BANDS if score >= cutoff)

nxt = make_stream()
records = [Score(st, sub, nxt(60, 100)) for st in students for sub in subjects]
records.append(Score(students[0], subjects[0], nxt(60, 100)))   
records[7] = records[7]._replace(score=None)                    

table([("No.", 4), ("Student", 14), ("Subject", 12), ("Score", 7), ("Category", 18)],
      [(n, r.student, r.subject, "MISSING" if r.score is None else r.score, categorize(r.score))
       for n, r in enumerate(records, 1)])

No.  | Student        | Subject      | Score   | Category          
-------------------------------------------------------------------
1    | PALAOAG_S1     | PROGRAMMING  | 68      | NEEDS IMPROVEMENT 
2    | PALAOAG_S1     | MATHEMATICS  | 62      | NEEDS IMPROVEMENT 
3    | PALAOAG_S1     | ELECTRONICS  | 77      | SATISFACTORY      
4    | PALAOAG_S1     | PHYSICS      | 80      | GOOD              
5    | PALAOAG_S1     | ENGLISH      | 75      | SATISFACTORY      
6    | PALAOAG_S2     | PROGRAMMING  | 99      | EXCELLENT         
7    | PALAOAG_S2     | MATHEMATICS  | 65      | NEEDS IMPROVEMENT 
8    | PALAOAG_S2     | ELECTRONICS  | MISSING | INCOMPLETE        
9    | PALAOAG_S2     | PHYSICS      | 85      | GOOD              
10   | PALAOAG_S2     | ENGLISH      | 65      | NEEDS IMPROVEMENT 
11   | PALAOAG_S3     | PROGRAMMING  | 97      | EXCELLENT         
12   | PALAOAG_S3     | MATHEMATICS  | 91      | EXCELLENT         
13   | PALAOAG_S3     | ELECTRONICS  | 65      |

In [3]:
organized = {}
for r in records:
    organized.setdefault(r.student, {}).setdefault(r.subject, []).append(r.score)

grid_rows = [(st, *["/".join("MISSING" if x is None else str(x)
                             for x in organized.get(st, {}).get(sub, [])) or "-"
                    for sub in subjects])
             for st in students]
table([("Student", 14)] + [(sub[:11], 12) for sub in subjects], grid_rows)

Student        | PROGRAMMING  | MATHEMATICS  | ELECTRONICS  | PHYSICS      | ENGLISH     
-----------------------------------------------------------------------------------------
PALAOAG_S1     | 68/75        | 62           | 77           | 80           | 75          
PALAOAG_S2     | 99           | 65           | MISSING      | 85           | 65          
PALAOAG_S3     | 97           | 91           | 65           | 68           | 63          
PALAOAG_S4     | 87           | 94           | 99           | 73           | 94          
PALAOAG_S5     | 85           | 79           | 94           | 97           | 92          


In [ ]:
pair_counts = Counter((r.student, r.subject) for r in records)
repeated    = {k: n for k, n in pair_counts.items() if n > 1}
incomplete  = [r for r in records if r.score is None]

for (st, sub), n in repeated.items():
    print(f"Repeated  : {st} / {sub} appears {n} times -> {organized[st][sub]}")
for r in incomplete:
    print(f"Incomplete: {r.student} / {r.subject} has a missing score")

seen, unique_records = set(), []
for r in records:
    key = (r.student, r.subject)
    if key not in seen:
        seen.add(key)
        unique_records.append(r)
valid = [r for r in unique_records if r.score is not None]
print(f"\nRecords: {len(records)} | unique: {len(unique_records)} | valid: {len(valid)}")

strong = lambda subject: {r.student for r in valid if r.subject == subject and r.score >= 80}
prog, maths = strong("PROGRAMMING"), strong("MATHEMATICS")
print("\nStudents with 80+ in PROGRAMMING:", sorted(prog))
print("Students with 80+ in MATHEMATICS:", sorted(maths))
print("Union                :", sorted(prog | maths))
print("Intersection         :", sorted(prog & maths))
print("Difference (P - M)   :", sorted(prog - maths))
print("Symmetric difference :", sorted(prog ^ maths))

with_missing = {r.student for r in incomplete}
print("Students with a complete record set:", sorted(set(students) - with_missing))

Repeated  : PALAOAG_S1 / PROGRAMMING appears 2 times -> [68, 75]
Incomplete: PALAOAG_S2 / ELECTRONICS has a missing score

Records: 26 | unique: 25 | valid: 24

Students with 80+ in PROGRAMMING: ['PALAOAG_S2', 'PALAOAG_S3', 'PALAOAG_S4', 'PALAOAG_S5']
Students with 80+ in MATHEMATICS: ['PALAOAG_S3', 'PALAOAG_S4']
Union                : ['PALAOAG_S2', 'PALAOAG_S3', 'PALAOAG_S4', 'PALAOAG_S5']
Intersection         : ['PALAOAG_S3', 'PALAOAG_S4']
Difference (P - M)   : ['PALAOAG_S2', 'PALAOAG_S5']
Symmetric difference : ['PALAOAG_S2', 'PALAOAG_S5']
Students with a complete record set: ['PALAOAG_S1', 'PALAOAG_S3', 'PALAOAG_S4', 'PALAOAG_S5']


In [ ]:
category_counts = Counter(categorize(r.score) for r in valid)
labels = [label for _, label in GRADE_BANDS]
table([("Category", 18), ("Count", 5)], [(label, category_counts[label]) for label in labels])  

Category           | Count
--------------------------
EXCELLENT          | 9    
GOOD               | 4    
SATISFACTORY       | 3    
NEEDS IMPROVEMENT  | 8    


In [6]:
by_student = {s: array("i", [r.score for r in valid if r.student == s]) for s in students}
by_subject = {s: array("i", [r.score for r in valid if r.subject == s]) for s in subjects}

def stats(arr):
    if not len(arr):
        return (0, 0, 0, 0.0)
    return (len(arr), max(arr), min(arr), round(mean(arr), 2))

print("Students")
table([("Student", 14), ("Subjects", 8), ("Highest", 7), ("Lowest", 7), ("Average", 8), ("Category", 18)],
      [(s, *stats(a), categorize(round(mean(a)))) for s, a in by_student.items()])
print("\nSubjects")
table([("Subject", 12), ("Students", 8), ("Highest", 7), ("Lowest", 7), ("Average", 8)],
      [(s, *stats(a)) for s, a in by_subject.items()])

Students
Student        | Subjects | Highest | Lowest  | Average  | Category          
-----------------------------------------------------------------------------
PALAOAG_S1     | 5        | 80      | 62      | 72.4     | NEEDS IMPROVEMENT 
PALAOAG_S2     | 4        | 99      | 65      | 78.5     | SATISFACTORY      
PALAOAG_S3     | 5        | 97      | 63      | 76.8     | SATISFACTORY      
PALAOAG_S4     | 5        | 99      | 73      | 89.4     | GOOD              
PALAOAG_S5     | 5        | 97      | 79      | 89.4     | GOOD              

Subjects
Subject      | Students | Highest | Lowest  | Average 
------------------------------------------------------
PROGRAMMING  | 5        | 99      | 68      | 87.2    
MATHEMATICS  | 5        | 94      | 62      | 78.2    
ELECTRONICS  | 4        | 99      | 65      | 83.75   
PHYSICS      | 5        | 97      | 68      | 80.6    
ENGLISH      | 5        | 94      | 63      | 77.8    


In [7]:
def rank_table(summary, name_title, width):
    queue = deque(sorted(summary.items(), key=lambda kv: mean(kv[1]), reverse=True))
    rows, rank = [], 1
    while queue:
        name, arr = queue.popleft()
        rows.append((rank, name, round(mean(arr), 2)))
        rank += 1
    table([("Rank", 4), (name_title, width), ("Average", 8)], rows)
    return rows

print("Students ranked by average")
student_rank = rank_table(by_student, "Student", 14)
print("\nSubjects ranked by average")
subject_rank = rank_table(by_subject, "Subject", 12)

Students ranked by average
Rank | Student        | Average 
--------------------------------
1    | PALAOAG_S4     | 89.4    
2    | PALAOAG_S5     | 89.4    
3    | PALAOAG_S2     | 78.5    
4    | PALAOAG_S3     | 76.8    
5    | PALAOAG_S1     | 72.4    

Subjects ranked by average
Rank | Subject      | Average 
------------------------------
1    | PROGRAMMING  | 87.2    
2    | ELECTRONICS  | 83.75   
3    | PHYSICS      | 80.6    
4    | MATHEMATICS  | 78.2    
5    | ENGLISH      | 77.8    


In [8]:
all_scores = array("i", [r.score for r in valid])
print(f"Total records generated : {len(records)}")
print(f"Valid unique scores     : {len(valid)}")
print(f"Repeated records        : {len(repeated)}")
print(f"Incomplete records      : {len(incomplete)}")
print(f"Class average           : {mean(all_scores):.2f}")
print(f"Top student             : {student_rank[0][1]} ({student_rank[0][2]})")
print(f"Lowest student          : {student_rank[-1][1]} ({student_rank[-1][2]})")
print(f"Strongest subject       : {subject_rank[0][1]} ({subject_rank[0][2]})")
print(f"Weakest subject         : {subject_rank[-1][1]} ({subject_rank[-1][2]})")

Total records generated : 26
Valid unique scores     : 24
Repeated records        : 1
Incomplete records      : 1
Class average           : 81.42
Top student             : PALAOAG_S4 (89.4)
Lowest student          : PALAOAG_S1 (72.4)
Strongest subject       : PROGRAMMING (87.2)
Weakest subject         : ENGLISH (77.8)
